<p style="text-align:center; font-size: 28px;">AI in Health Technologies (2026)</p>
<p style="text-align:center; font-size: 28px;">Project: Source or clinical content?</p>
<p style="text-align:center; font-size: 20px;">Alexandra Dahlberg, Heidi Tarkiainen, Joonatan Syrjänen</p>

### Table of Contents

* [1. Introduction](#section_1)
* [2. Setup](#section_2)
* [3. Data](#section_3)
* [4. Embeddings](#section_4)
* [5. Pictures](#section_5)
    * [5.1. PCA](#section_5_1)
    * [5.2. t-SNE](#section_5_2)
    * [5.3. UMAP](#section_5_3)
    * [5.4. Notes vs imaging texts](#section_5_4)
* [6. Q1: Do the notes separate by text type?](#section_6)
* [7. Classifiers (Joonatan)](#section_7)
* [8. Q2: Does the picture predict how hard a diagnosis is?](#section_8)
* [9. Seeds and UMAP settings](#section_9)
* [10. Note type and diagnosis mixed up?](#section_10)
* [11. FinBERT layers (Acamedic only)](#section_11)
* [12. Clinical review](#section_12)
* [13. Package versions](#section_13)

## 1. Introduction <a class="anchor" id="section_1"></a>

A language model turns each clinical note into an embedding, a long list of numbers. PCA, t-SNE and UMAP squeeze these into a 2-D picture where similar notes end up close together. In earlier work on lung imaging reports, our group saw that these pictures split clearly by note type but not by cancer diagnosis. That worries us: if a model can tell what kind of note it is reading, anything trained on top of it might learn the note type instead of the disease.

So we looked at vascular surgery notes from one EHR (Apotti) and asked two questions:

- **Q1:** Do the notes separate by text type (note type, specialty, year) rather than by what they say?
- **Q2:** Do they separate by diagnosis, and does the picture tell us how hard a diagnosis classifier will find the task?

We did not want to trust the pictures alone, so we checked each one with numbers in the full embedding space.

We built everything on synthetic notes first, with structure planted on purpose, and then ran the same notebook on the real notes in HUS Acamedic. Only numbers and figures left Acamedic, never text.

**Running it.** On the synthetic data the notebook runs from top to bottom without Ollama, because the embeddings are saved in `embeddings/`. In Colab, the first code cell gets the repo and installs `umap-learn`.

## 2. Setup <a class="anchor" id="section_2"></a>

In [ ]:
# Packages
import os
import sys
import warnings
from importlib.metadata import version
warnings.filterwarnings("ignore", category=FutureWarning) # Notices about future package versions, they did not change the results

# In Colab we first had to get the repo and umap-learn
if "google.colab" in sys.modules:
    os.system("pip install umap-learn")
    os.system("git clone https://github.com/joonatansyrjanen/medical-text-umap")
    os.chdir("medical-text-umap")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
import umap # pip install umap-learn

import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel

In [ ]:
# Synthetic data here, real data in Acamedic
data_source = "synthetic" # We changed this to "real" in Acamedic

if data_source == "synthetic":
    data_folder = "mock_data"
else:
    data_folder = "/path/to/acamedic/export"

# Embedding model, run through Ollama
model_name = "bge-m3"
ollama_url = "http://localhost:11434" # In Acamedic this was the address of the Azure ML compute
max_tokens = 8192 # BGE-M3 reads up to 8192 tokens, but Ollama cuts at 2048 unless we tell it otherwise

# FinBERT, only in Acamedic where we had it
run_finbert = False
finbert_path = "TurkuNLP/bert-base-finnish-cased-v1"
main_layer = 12 # We decided beforehand to use the last layer, the others are only for comparison

# Settings we fixed before looking at the real data
seed = 42
k = 15 # Neighbours in the kNN agreement
min_characters = 40 # Shorter notes were dropped
min_patients_per_code = 10 # Rarer ICD codes were left out of the diagnosis analysis
number_of_folds = 5 # Cross-validation folds, grouped by patient
epochs = 200 # Classifier training rounds, all training notes at once
learning_rate = 0.001
weight_decay = 0.0001 # Keeps the classifier weights small
hidden_units = 256 # Size of the MLP hidden layer
tfidf_words = 5000 # Most common words kept for the TF-IDF baseline
device = "cuda" if torch.cuda.is_available() else "cpu" # The classifiers ran on the GPU when there was one
seeds = [0, 1, 2, 3, 4] # Random seeds for t-SNE and UMAP, seed 0 is the main picture
umap_neighbours = [5, 15, 50] # UMAP settings, 15 is the main picture
texts_per_table = 2500 # At most this many texts per table in the notes vs imaging picture, like Rami
number_of_clusters = 8 # For the clinical review, we set this after looking at the UMAP
notes_per_cluster = 5 # Notes the clinician read from each cluster

# Labels we coloured the pictures by
labels_to_check = ["note_type", "specialty", "year", "length_quartile", "diagnosis_group", "diagnosis_code"]

# Rami's colours from the lung imaging figures, plus a few lighter ones for labels with many classes
palette = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#EECA3B", "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC",
           "#9ECAE9", "#FFBF79", "#88D27A", "#83BCB6", "#FF9D98", "#D6A5C9"]
source_colours = {"apotti_kuvantaminen_lausunto": "#4C78A8", # Imaging reports, blue like Rami's
                  "apotti_kuvantaminen_tutkimus": "#F58518", # Imaging requests, orange
                  "apotti_potilaskertomus": "#54A24B"} # Patient record notes, green

# Figures and tables go here
results_folder = "results/" + model_name
os.makedirs(results_folder, exist_ok=True)

## 3. Data <a class="anchor" id="section_3"></a>

The texts are the free-text patient record entries (potilaskertomus). Each note got the main ICD-10 diagnosis of its care episode, joined through the episode ID.

Every note starts with a line giving the writer's role, name and time. Staff names alone give away the hospital, so we removed that line before embedding.

In [ ]:
# Notes and diagnoses, both tables use ; as the separator
notes = pd.read_csv(data_folder + "/Apotti_potilaskertomus.csv", sep=";")
diagnoses = pd.read_csv(data_folder + "/Apotti_diagnoosi.csv", sep=";")

# Main diagnosis of each care episode, joined to the notes through the episode ID
main_diagnoses = diagnoses[diagnoses["paadiagnoosi"] == True]
main_diagnoses = main_diagnoses.drop_duplicates(subset="shjakso_id") # A few episodes had the same code twice
notes = notes.merge(main_diagnoses[["shjakso_id", "diagnoosi_koodi"]], on="shjakso_id", how="left")

# Shorter English names for the columns we used
notes["note_id"] = notes["merkinta_id"]
notes["patient_id"] = notes["henkilotunnus"]
notes["note_type"] = notes["merkinta_valilehti_selite"] # E.g. Päivittäismerkintä
notes["specialty"] = notes["toimipiste_eala_selite"] # Specialty of the writing unit
notes["year"] = notes["merkinta_luonti_hetki"].str[:4] # E.g. 2023
notes["diagnosis_code"] = notes["diagnoosi_koodi"] # E.g. I70.2
notes["diagnosis_group"] = notes["diagnoosi_koodi"].str[:3] # E.g. I70

# The text without its first line (role, name and time)
notes["text"] = notes["merkinta_teksti"].str.split("\n", n=1).str[1]
notes["text"] = notes["text"].fillna("") # Notes with nothing after the first line

Very short notes, like "Uusi aika annettu.", say almost nothing, so we dropped them. Next we split note length into four equal groups. That way we could see whether a picture just sorts short notes from long ones (the proposal feedback suggested length as a surface baseline). ICD codes with fewer than 10 patients were left out of the diagnosis analysis, but their notes stayed in.

In [ ]:
# Very short notes dropped
notes["characters"] = notes["text"].str.len()
print("Notes shorter than %d characters dropped: %d"
      % (min_characters, np.sum(notes["characters"] < min_characters)))
notes = notes[notes["characters"] >= min_characters]
notes = notes.reset_index(drop=True) # Row n of notes is row n of the embeddings

# Note length in four equal groups
notes["length_quartile"] = pd.qcut(notes["characters"], 4, labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"]).astype(str)

# Rare ICD codes and groups left out of the diagnosis analysis
patients_per_code = notes.groupby("diagnosis_code")["patient_id"].nunique()
rare_codes = patients_per_code[patients_per_code < min_patients_per_code].index
notes.loc[notes["diagnosis_code"].isin(rare_codes), "diagnosis_code"] = np.nan

patients_per_group = notes.groupby("diagnosis_group")["patient_id"].nunique()
rare_groups = patients_per_group[patients_per_group < min_patients_per_code].index
notes.loc[notes["diagnosis_group"].isin(rare_groups), "diagnosis_group"] = np.nan

# Most notes from one patient, needed for the kNN agreement in Section 6
most_notes_per_patient = notes.groupby("patient_id").size().max()

# Numbers for Table 1
print("Notes: %d"
      % notes.shape[0])
print("Patients: %d"
      % notes["patient_id"].nunique())
print("ICD-10 codes kept: %d"
      % notes["diagnosis_code"].nunique())
print("Most notes from one patient: %d"
      % most_notes_per_patient)
for label in ["note_type", "specialty", "year", "diagnosis_group"]:
    print()
    print(notes[label].value_counts())

In the earlier lung work, notes and radiology reports ended up far apart in the pictures. To see the same thing in our data, we also took the imaging reports (lausunto) and imaging requests (tutkimus). As in Rami's figures, we used at most 2500 texts per table.

In [ ]:
# Imaging reports and requests
reports_table = pd.read_csv(data_folder + "/Apotti_kuvantaminen_lausunto.csv", sep=";")
requests_table = pd.read_csv(data_folder + "/Apotti_kuvantaminen_tutkimus.csv", sep=";")

reports = pd.DataFrame({"text_id": reports_table["ac_numero"].astype(str) + "_lausunto",
                        "patient_id": reports_table["henkilotunnus"],
                        "source": "apotti_kuvantaminen_lausunto",
                        "text": reports_table["lausunto_teksti"]})
imaging_requests = pd.DataFrame({"text_id": requests_table["maarays_id"].astype(str) + "_pyynto",
                                 "patient_id": requests_table["henkilotunnus"],
                                 "source": "apotti_kuvantaminen_tutkimus",
                                 "text": requests_table["pyynto_teksti"]})

# At most texts_per_table from each table
if reports.shape[0] > texts_per_table:
    reports = reports.sample(n=texts_per_table, random_state=seed)
if imaging_requests.shape[0] > texts_per_table:
    imaging_requests = imaging_requests.sample(n=texts_per_table, random_state=seed)

imaging = pd.concat([reports, imaging_requests], ignore_index=True)
imaging = imaging[imaging["text"].str.len() >= min_characters] # Same rule as for the notes
imaging = imaging.reset_index(drop=True)

print(imaging["source"].value_counts())

## 4. Embeddings <a class="anchor" id="section_4"></a>

We used BGE-M3, a multilingual model that covers Finnish and reads up to 8192 tokens. We ran it through Ollama, both on our own computers and in Acamedic, so the same code worked in both places. Ollama gives one vector of 1024 numbers per note.

Embedding takes a while, so we saved the vectors in `embeddings/` and loaded them on later runs. To make them again, install Ollama, run `ollama pull bge-m3` in a terminal and delete the `.npy` file.

In [ ]:
def embed(text):
    """Sends one text to Ollama and gets back its embedding and the number of tokens it read."""
    response = requests.post(ollama_url + "/api/embed", json={
        "model": model_name,
        "input": text,
        "options": {"num_ctx": max_tokens, "num_batch": max_tokens}, # Without these Ollama cut every text at 2048 tokens
    })
    answer = response.json()
    return answer["embeddings"][0], answer["prompt_eval_count"]


# Notes, embedded once and then loaded from the file
notes_file = "embeddings/" + model_name + "_no_header.npy"
notes_tokens_file = "embeddings/" + model_name + "_no_header_labels.csv"

if os.path.exists(notes_file):
    embeddings = np.load(notes_file)
    saved = pd.read_csv(notes_tokens_file)
    assert saved["note_id"].tolist() == notes["note_id"].tolist(), "The saved embeddings are for other notes, delete the file and embed again"
    notes["tokens"] = saved["tokens"]
    print("Loaded %d embeddings from %s"
          % (embeddings.shape[0], notes_file))
else:
    # First we checked that Ollama really read 8192 tokens, otherwise long notes get cut without any warning
    vector, tokens_read = embed("Potilas voi hyvin. " * 5000)
    print("Tokens read from a very long text: %d (should be %d)"
          % (tokens_read, max_tokens))

    # Then every note, one at a time
    embeddings = []
    tokens = []
    for text in notes["text"]:
        vector, tokens_read = embed(text)
        embeddings.append(vector)
        tokens.append(tokens_read)
        if len(embeddings) % 1000 == 0:
            print("Embedded %d of %d notes"
                  % (len(embeddings), notes.shape[0]))
    embeddings = np.array(embeddings)
    notes["tokens"] = tokens
    np.save(notes_file, embeddings)
    notes[["note_id", "tokens"]].to_csv(notes_tokens_file, index=False)

print("Embeddings: %d notes, %d numbers each"
      % embeddings.shape)

In [ ]:
# Imaging reports and requests, the same way
imaging_file = "embeddings/" + model_name + "_imaging.npy"
imaging_tokens_file = "embeddings/" + model_name + "_imaging_labels.csv"

if os.path.exists(imaging_file):
    imaging_embeddings = np.load(imaging_file)
    saved = pd.read_csv(imaging_tokens_file)
    assert saved["text_id"].tolist() == imaging["text_id"].tolist(), "The saved embeddings are for other texts, delete the file and embed again"
    print("Loaded %d embeddings from %s"
          % (imaging_embeddings.shape[0], imaging_file))
else:
    imaging_embeddings = []
    tokens = []
    for text in imaging["text"]:
        vector, tokens_read = embed(text)
        imaging_embeddings.append(vector)
        tokens.append(tokens_read)
    imaging_embeddings = np.array(imaging_embeddings)
    imaging["tokens"] = tokens
    np.save(imaging_file, imaging_embeddings)
    imaging[["text_id", "tokens"]].to_csv(imaging_tokens_file, index=False)
    print("Embedded %d imaging texts"
          % imaging_embeddings.shape[0])

The feedback asked how we handled notes over 512 tokens. BGE-M3 reads up to 8192 tokens, so with BGE-M3 nothing was cut. FinBERT (Section 11) reads only 512, so there a longer note was cut to its first 512 tokens.

In [ ]:
# How long the notes were, counted with the BGE-M3 tokenizer
print("Longest note: %d tokens"
      % notes["tokens"].max())
print("Notes cut at %d tokens by BGE-M3: %d"
      % (max_tokens, np.sum(notes["tokens"] >= max_tokens)))
print("Notes over 512 tokens, which FinBERT would cut: %d (%.1f %%)"
      % (np.sum(notes["tokens"] > 512), 100 * np.mean(notes["tokens"] > 512)))

## 5. Pictures <a class="anchor" id="section_5"></a>

We made the pictures with PCA, t-SNE and UMAP and coloured each one by every label. PCA is the simple linear one. t-SNE and UMAP try to keep notes that are close in the full space close in the picture too. Their axes have no units, and the distance between two far-apart clusters means very little.

We drew all pictures the same way as Rami's lung figures: his colours, the biggest class first and the number of notes in the legend. This function draws one picture.

In [ ]:
def plot_by_label(ax, points, labels, label_name, title):
    """Draws the 2-D points as a square picture, one colour per class, like Rami's figures."""
    labels = pd.Series(list(labels)) # Plain 0, 1, 2... index so it matches the rows of points

    # Text sources keep Rami's colours, other labels get his palette with the biggest class first
    if label_name == "source":
        classes = [name for name in source_colours if name in labels.values]
        colours = source_colours
    else:
        classes = list(labels.value_counts().index)
        if label_name in ["year", "length_quartile"]:
            classes = sorted(classes) # In order rather than by size
        colours = {}
        for index in range(len(classes)):
            colours[classes[index]] = palette[index % len(palette)]

    # Notes without a label (rare ICD codes) in light grey underneath
    missing = labels.isnull().to_numpy()
    if missing.sum() > 0:
        ax.scatter(points[missing, 0], points[missing, 1], s=4, color="#DDDDDD", label="no label (%d)" % missing.sum())

    # Then one class at a time, with the number of notes in the legend
    for name in classes:
        in_class = (labels == name).to_numpy()
        ax.scatter(points[in_class, 0], points[in_class, 1], s=4, alpha=0.7, color=colours[name],
                   label="%s (%d)" % (name, in_class.sum()))

    ax.set_title(title)
    ax.set_xticks([]) # No units on these axes
    ax.set_yticks([])
    ax.set_box_aspect(1) # Square picture
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, markerscale=3)

### 5.1. PCA <a class="anchor" id="section_5_1"></a>

In [ ]:
# PCA to 2-D
pca = PCA(n_components=2)
pca_points = pca.fit_transform(embeddings)
print("Variance explained by the two components: %.1f %%"
      % (100 * np.sum(pca.explained_variance_ratio_)))

# One picture per label
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.subplots_adjust(hspace=0.45) # Room for the legends between the rows
for index in range(len(labels_to_check)):
    label = labels_to_check[index]
    plot_by_label(axes.flat[index], pca_points, notes[label], label, label)
fig.suptitle("PCA")
fig.savefig(results_folder + "/pca.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2. t-SNE <a class="anchor" id="section_5_2"></a>

We ran t-SNE with five random seeds (cosine distance, perplexity 30, starting from the PCA layout). Seed 0 is the picture we show; the other seeds come back in Section 9.

In [ ]:
# t-SNE, one run per seed
tsne_points = {}
for projection_seed in seeds:
    tsne = TSNE(metric="cosine", perplexity=30, init="pca", random_state=projection_seed)
    tsne_points[projection_seed] = tsne.fit_transform(embeddings)
    print("t-SNE seed %d done"
          % projection_seed)

# One picture per label, seed 0
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.subplots_adjust(hspace=0.45) # Room for the legends between the rows
for index in range(len(labels_to_check)):
    label = labels_to_check[index]
    plot_by_label(axes.flat[index], tsne_points[0], notes[label], label, label)
fig.suptitle("t-SNE, seed 0")
fig.savefig(results_folder + "/tsne.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3. UMAP <a class="anchor" id="section_5_3"></a>

UMAP ran with cosine distance and `min_dist` 0.1, starting from the PCA layout. The default start failed on the exact duplicate notes that copy-paste creates. We tried `n_neighbors` 5, 15 and 50 with the same five seeds; `n_neighbors` 15 with seed 0 is the main picture.

In [ ]:
# UMAP, every setting with every seed
umap_points = {}
for n_neighbors in umap_neighbours:
    for projection_seed in seeds:
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, metric="cosine", init="pca", random_state=projection_seed, n_jobs=1) # One thread, because we fixed the seed
        umap_points[(n_neighbors, projection_seed)] = reducer.fit_transform(embeddings)
    print("UMAP n_neighbors %d done"
          % n_neighbors)
main_umap = umap_points[(15, 0)] # The main picture

# One picture per label
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.subplots_adjust(hspace=0.45) # Room for the legends between the rows
for index in range(len(labels_to_check)):
    label = labels_to_check[index]
    plot_by_label(axes.flat[index], main_umap, notes[label], label, label)
fig.suptitle("UMAP, n_neighbors 15, seed 0")
fig.savefig(results_folder + "/umap.png", dpi=150, bbox_inches="tight")
plt.show()

# The coordinates of the main pictures, these were allowed to leave Acamedic
coordinates = pd.DataFrame({"note_id": notes["note_id"],
                            "pca_x": pca_points[:, 0], "pca_y": pca_points[:, 1],
                            "tsne_x": tsne_points[0][:, 0], "tsne_y": tsne_points[0][:, 1],
                            "umap_x": main_umap[:, 0], "umap_y": main_umap[:, 1]})
coordinates.to_csv(results_folder + "/coordinates.csv", index=False)

### 5.4. Notes vs imaging texts <a class="anchor" id="section_5_4"></a>

The same picture as Rami's lung imaging figure, but inside one EHR: patient record notes, imaging reports and imaging requests embedded with the same model, in his colours. We took 2500 random notes, so that the notes did not swamp the imaging texts.

In [ ]:
# 2500 random notes and the imaging texts together
note_sample = notes.sample(n=min(texts_per_table, notes.shape[0]), random_state=seed)
source_embeddings = np.vstack([embeddings[note_sample.index], imaging_embeddings])
source_labels = ["apotti_potilaskertomus"] * note_sample.shape[0] + imaging["source"].tolist()
source_patients = note_sample["patient_id"].tolist() + imaging["patient_id"].tolist()

# UMAP and t-SNE with the main settings
source_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", init="pca", random_state=0, n_jobs=1).fit_transform(source_embeddings)
source_tsne = TSNE(metric="cosine", perplexity=30, init="pca", random_state=0).fit_transform(source_embeddings)

# Side by side, like Rami's figure
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
plot_by_label(axes[0], source_umap, source_labels, "source", "UMAP")
plot_by_label(axes[1], source_tsne, source_labels, "source", "t-SNE")
fig.savefig(results_folder + "/notes_vs_imaging.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Q1: Do the notes separate by text type? <a class="anchor" id="section_6"></a>

A picture can show clusters that are not really there, or hide ones that are. So next we wanted to check each picture with numbers, and we did it in the full 1024-number space as well as in 2-D.

**kNN label agreement.** For every note we looked at its 15 nearest notes and counted how many had the same label. We skipped notes from the same patient: copy-paste makes a patient's notes almost identical, and that would make every patient-level label, like the diagnosis, look separated. Big classes agree a lot just by chance, so we compared the agreement with the chance level, the sum of the squared class shares.

If the agreement is about the same in 2-D as in the full space, the picture is honest. If it is much lower, the picture hides structure.

In [ ]:
def knn_agreement(points, labels, patients, metric):
    """For every note, the share of its k nearest notes from other patients that have the same label."""
    # Notes without a label (rare ICD codes) were left out
    has_label = pd.Series(list(labels)).notnull().to_numpy()
    points = points[has_label]
    labels = np.array(list(labels), dtype=object)[has_label]
    patients = np.array(list(patients))[has_label]

    # We asked for extra neighbours, because the same patient's notes are skipped
    number_of_neighbours = min(len(labels), k + most_notes_per_patient)
    nearest = NearestNeighbors(n_neighbors=number_of_neighbours, metric=metric)
    nearest.fit(points)
    distances, neighbours = nearest.kneighbors(points)

    agreement = np.zeros(len(labels))
    for n in range(len(labels)):
        other_patients = neighbours[n][patients[neighbours[n]] != patients[n]] # Skips the note itself and its patient
        agreement[n] = np.mean(labels[other_patients[:k]] == labels[n])
    return agreement, labels


def chance_level(labels):
    """Agreement we would get by chance: the sum of the squared class shares."""
    shares = pd.Series(labels).value_counts(normalize=True)
    return np.sum(shares ** 2)

In [ ]:
# Agreement for every label, in the full space and in the three main pictures
rows = []
for label in labels_to_check:
    agreement_full, used_labels = knn_agreement(embeddings, notes[label], notes["patient_id"], "cosine")
    agreement_pca, used_labels = knn_agreement(pca_points, notes[label], notes["patient_id"], "euclidean")
    agreement_tsne, used_labels = knn_agreement(tsne_points[0], notes[label], notes["patient_id"], "euclidean")
    agreement_umap, used_labels = knn_agreement(main_umap, notes[label], notes["patient_id"], "euclidean")
    rows.append([label, chance_level(used_labels), np.mean(agreement_full),
                 np.mean(agreement_pca), np.mean(agreement_tsne), np.mean(agreement_umap)])

knn_table = pd.DataFrame(rows, columns=["label", "chance", "full", "pca", "tsne", "umap"])
knn_table.to_csv(results_folder + "/knn_agreement.csv", index=False)
knn_table.round(2)

In the synthetic data we wrote each note type in its own style on purpose. So if the pipeline works, note type should come out far above chance here, both in the full space and in the pictures.

Then the same check for the notes vs imaging texts from 5.4.

In [ ]:
# Agreement for the text source, full space and the two pictures
agreement_full, used_labels = knn_agreement(source_embeddings, source_labels, source_patients, "cosine")
agreement_umap, used_labels = knn_agreement(source_umap, source_labels, source_patients, "euclidean")
agreement_tsne, used_labels = knn_agreement(source_tsne, source_labels, source_patients, "euclidean")
print("Notes vs imaging texts, chance %.2f: full space %.2f, UMAP %.2f, t-SNE %.2f"
      % (chance_level(used_labels), np.mean(agreement_full), np.mean(agreement_umap), np.mean(agreement_tsne)))

We also computed silhouette scores, as planned in the proposal. They go from -1 to 1, and around 0 means the classes overlap. Unlike the kNN agreement, they do not skip the same patient's notes.

In [ ]:
# Silhouette for every label, full space and the three main pictures
rows = []
for label in labels_to_check:
    has_label = notes[label].notnull().to_numpy()
    labels = notes[label][has_label]
    rows.append([label,
                 silhouette_score(embeddings[has_label], labels, metric="cosine"),
                 silhouette_score(pca_points[has_label], labels),
                 silhouette_score(tsne_points[0][has_label], labels),
                 silhouette_score(main_umap[has_label], labels)])

silhouette_table = pd.DataFrame(rows, columns=["label", "full", "pca", "tsne", "umap"])
silhouette_table.to_csv(results_folder + "/silhouette.csv", index=False)
silhouette_table.round(2)

## 7. Classifiers <a class="anchor" id="section_7"></a>

Here we trained simple classifiers in PyTorch on the full embeddings: one `nn.Linear` layer and a small MLP. As surface baselines (from the feedback), the same model was trained on note length alone and on TF-IDF of the 5000 most common words. Cross-validation had five folds grouped by patient, so no patient was in both training and test folds.

What is already in memory: `embeddings` (one row per note), and `notes` with the labels, `text` (for TF-IDF), `characters` and `tokens` (for the length baseline) and `patient_id`.

For Q2 in Section 8, save the AUROC per ICD code to `results/bge-m3/classifier_auroc_per_code.csv`, with the columns `diagnosis_code` and `auroc`.

In [ ]:
# Notes with a diagnosis code, the rare codes were already left out in Section 3
has_code = notes["diagnosis_code"].notnull().to_numpy()
classes = sorted(notes["diagnosis_code"][has_code].unique())
y = notes["diagnosis_code"][has_code].map({code: i for i, code in enumerate(classes)}).to_numpy()
groups = notes["patient_id"][has_code].to_numpy()

# The same folds for every model, grouped by patient and stratified by code
folds = list(StratifiedGroupKFold(n_splits=number_of_folds, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, groups))

print("Notes: %d, patients: %d, ICD codes: %d"
      % (len(y), len(set(groups)), len(classes)))

In [ ]:
def train_and_predict(X_train, y_train, X_test, hidden=None):
    """Trains a linear layer, or an MLP if hidden is given, and returns the class probabilities for X_test."""
    torch.manual_seed(seed) # Same starting weights in every fold

    # Standardised with the training notes only, so nothing leaks from the test fold
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-8 # Avoids dividing by 0 for constant columns
    X_train = torch.tensor((X_train - mean) / std, dtype=torch.float32, device=device)
    X_test = torch.tensor((X_test - mean) / std, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, device=device)

    if hidden is None:
        model = nn.Linear(X_train.shape[1], len(classes))
    else:
        model = nn.Sequential(nn.Linear(X_train.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, len(classes)))
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    for epoch in range(epochs): # All training notes at once, if the real data is too large we need DataLoader
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(X_train), y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        return torch.softmax(model(X_test), dim=1).cpu().numpy()

In [ ]:
def cross_validate(features, hidden=None):
    """Class probabilities for every note, each from a model that did not see the note's patient in training."""
    probabilities = np.zeros((len(y), len(classes)))
    for train, test in folds:
        probabilities[test] = train_and_predict(features[train], y[train], features[test], hidden)
    return probabilities


# Out-of-fold probabilities for every model
probabilities = {}
probabilities["Embeddings, linear"] = cross_validate(embeddings[has_code])
probabilities["Embeddings, MLP"] = cross_validate(embeddings[has_code], hidden=hidden_units)
lengths = np.log(notes[["characters", "tokens"]][has_code].to_numpy()) # Note length in characters and BGE-M3 tokens
probabilities["Length"] = cross_validate(lengths)

# TF-IDF learned its vocabulary inside each fold, from the training notes only
texts = notes["text"][has_code].to_numpy()
probabilities["TF-IDF"] = np.zeros((len(y), len(classes)))
for train, test in folds:
    vectorizer = TfidfVectorizer(max_features=tfidf_words, dtype=np.float32)
    X_train = vectorizer.fit_transform(texts[train]).toarray() # Dense, about 2 GB per 100 000 notes, switch to sparse if memory runs out
    X_test = vectorizer.transform(texts[test]).toarray()
    probabilities["TF-IDF"][test] = train_and_predict(X_train, y[train], X_test)

In [ ]:
# One-vs-rest AUROC per ICD code at note level, from the test folds of all five splits together
rows = []
for index in range(len(classes)):
    is_code = y == index
    rows.append([classes[index]] + [roc_auc_score(is_code, probabilities[model][:, index]) for model in probabilities])

classifier_table = pd.DataFrame(rows, columns=["diagnosis_code"] + list(probabilities))
classifier_table.to_csv(results_folder + "/classifier_auroc.csv", index=False)

# Section 8 uses the linear model on the embeddings, the main classifier
classifier_table.rename(columns={"Embeddings, linear": "auroc"})[["diagnosis_code", "auroc"]].to_csv(
    results_folder + "/classifier_auroc_per_code.csv", index=False)

print("Mean AUROC over the codes:")
print(classifier_table.mean(numeric_only=True).round(2))
classifier_table.round(2)

## 8. Q2: Does the picture predict how hard a diagnosis is? <a class="anchor" id="section_8"></a>

Following the feedback, we used the finer ICD-10 codes instead of the four groups, so that the rank correlation has more than four points.

For each code we measured how separated it looks in the main UMAP, and put that next to how well the classifier did on it (AUROC from Section 7). We used agreement above chance, where 0 means no better than chance and 1 means all 15 neighbours have the same code. The plain agreement goes up just because a code is common, so on its own it would not work for comparing codes.

If the codes line up, the picture works as a cheap check of how hard a diagnosis task will be. With 10 or more codes we report the Spearman correlation; with fewer we only describe it.

In [ ]:
# Agreement per ICD code, full space and main UMAP
agreement_full, codes = knn_agreement(embeddings, notes["diagnosis_code"], notes["patient_id"], "cosine")
agreement_umap, codes = knn_agreement(main_umap, notes["diagnosis_code"], notes["patient_id"], "euclidean")

rows = []
for code in sorted(set(codes)):
    is_code = codes == code
    chance = np.mean(is_code) # Share of notes with this code
    full_score = (np.mean(agreement_full[is_code]) - chance) / (1 - chance) # Agreement above chance
    umap_score = (np.mean(agreement_umap[is_code]) - chance) / (1 - chance)
    patients = notes.loc[notes["diagnosis_code"] == code, "patient_id"].nunique()
    rows.append([code, patients, np.sum(is_code), full_score, umap_score])

code_table = pd.DataFrame(rows, columns=["diagnosis_code", "patients", "notes", "full", "umap"])

# Classifier AUROC per code from Section 7, if it is there
auroc_file = results_folder + "/classifier_auroc_per_code.csv"
if os.path.exists(auroc_file):
    auroc_table = pd.read_csv(auroc_file)
    code_table = code_table.merge(auroc_table[["diagnosis_code", "auroc"]], on="diagnosis_code", how="left")
else:
    print("No classifier results yet, they come from Section 7")

code_table.to_csv(results_folder + "/per_code.csv", index=False)
code_table.round(2)

In [ ]:
# Picture vs classifier, one dot per ICD code, dot size by the number of patients
if "auroc" in code_table.columns:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(code_table["umap"], code_table["auroc"], s=3 * code_table["patients"], color=palette[0], alpha=0.7)
    for index in range(code_table.shape[0]):
        ax.annotate(code_table["diagnosis_code"][index], (code_table["umap"][index], code_table["auroc"][index]))
    ax.set_xlabel("kNN agreement above chance in the UMAP")
    ax.set_ylabel("Classifier AUROC")
    ax.set_box_aspect(1)
    fig.savefig(results_folder + "/picture_vs_classifier.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Spearman correlation, only with 10 or more codes
    if code_table.shape[0] >= 10:
        correlation, p_value = stats.spearmanr(code_table["umap"], code_table["auroc"])
        print("Spearman correlation %.2f, p = %.3g, %d codes"
              % (correlation, p_value, code_table.shape[0]))
    else:
        print("Only %d codes, so we describe this instead of giving a correlation"
              % code_table.shape[0])

## 9. Seeds and UMAP settings <a class="anchor" id="section_9"></a>

We did not want any conclusion to rest on one picture. So we repeated the kNN agreement for note type (Q1) and diagnosis code (Q2) over every t-SNE seed and every UMAP setting and seed, and compared them with PCA and the full space.

In [ ]:
# Agreement in every t-SNE and UMAP run
rows = []
for label in ["note_type", "diagnosis_code"]:
    for projection_seed in seeds:
        agreement, used_labels = knn_agreement(tsne_points[projection_seed], notes[label], notes["patient_id"], "euclidean")
        rows.append([label, "t-SNE", projection_seed, np.mean(agreement)])
    for n_neighbors in umap_neighbours:
        for projection_seed in seeds:
            agreement, used_labels = knn_agreement(umap_points[(n_neighbors, projection_seed)], notes[label], notes["patient_id"], "euclidean")
            rows.append([label, "UMAP n_neighbors %d" % n_neighbors, projection_seed, np.mean(agreement)])

seed_table = pd.DataFrame(rows, columns=["label", "projection", "seed", "agreement"])
seed_table.to_csv(results_folder + "/seeds_and_settings.csv", index=False)

# Mean, lowest and highest over the five seeds
print(seed_table.groupby(["label", "projection"])["agreement"].agg(["mean", "min", "max"]).round(2))

# PCA and the full space for comparison, from Section 6
print()
print(knn_table[knn_table["label"].isin(["note_type", "diagnosis_code"])][["label", "chance", "full", "pca"]].round(2))

In [ ]:
# The three UMAP settings side by side, coloured by note type
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for index in range(len(umap_neighbours)):
    n_neighbors = umap_neighbours[index]
    plot_by_label(axes[index], umap_points[(n_neighbors, 0)], notes["note_type"], "note_type", "UMAP n_neighbors %d" % n_neighbors)
fig.savefig(results_folder + "/umap_settings.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Note type and diagnosis mixed up? <a class="anchor" id="section_10"></a>

Some note types are much more common for some diagnoses, for example operation notes for patients who were operated on. A split by note type could then partly be a split by diagnosis, and the other way round. So we checked note type within each diagnosis group, and diagnosis code within each note type. We only used groups with at least 10 patients.

In [ ]:
rows = []

# Note type within each diagnosis group
for group in sorted(notes["diagnosis_group"].dropna().unique()):
    in_group = (notes["diagnosis_group"] == group).to_numpy()
    if notes["patient_id"][in_group].nunique() < min_patients_per_code:
        continue
    agreement, used_labels = knn_agreement(embeddings[in_group], notes["note_type"][in_group], notes["patient_id"][in_group], "cosine")
    rows.append(["note type", "within " + group, len(used_labels), chance_level(used_labels), np.mean(agreement)])

# Diagnosis code within each note type
for note_type in sorted(notes["note_type"].unique()):
    in_type = (notes["note_type"] == note_type).to_numpy()
    if notes["patient_id"][in_type].nunique() < min_patients_per_code:
        continue
    agreement, used_labels = knn_agreement(embeddings[in_type], notes["diagnosis_code"][in_type], notes["patient_id"][in_type], "cosine")
    rows.append(["diagnosis code", "within " + note_type, len(used_labels), chance_level(used_labels), np.mean(agreement)])

mixed_table = pd.DataFrame(rows, columns=["label", "subset", "notes", "chance", "agreement"])
mixed_table.to_csv(results_folder + "/note_type_and_diagnosis.csv", index=False)
mixed_table.round(2)

## 11. FinBERT layers (Acamedic only) <a class="anchor" id="section_11"></a>

Through Ollama we only get the final vector of BGE-M3. To see in which layer the note type shows up, we also used FinBERT, which gives every layer. FinBERT reads at most 512 tokens, so we cut each note to its first 512 tokens, and we mean pooled the token vectors of each layer. We had decided beforehand that the last layer (12) is the main one; the others are only for comparison.

This only runs with `run_finbert = True`.

In [ ]:
if run_finbert:
    tokenizer = AutoTokenizer.from_pretrained(finbert_path)
    finbert = AutoModel.from_pretrained(finbert_path, output_hidden_states=True) # Gives every layer, not just the last
    finbert.eval()

    # Mean of the token vectors in every layer, for every note
    layer_embeddings = np.zeros((13, notes.shape[0], 768)) # 13 layers (input and 1-12), 768 numbers each
    for n in range(notes.shape[0]):
        encoded = tokenizer(notes["text"][n], truncation=True, max_length=512, return_tensors="pt") # First 512 tokens
        with torch.no_grad():
            output = finbert(**encoded)
        for layer in range(13):
            layer_embeddings[layer, n] = output.hidden_states[layer][0].mean(dim=0).numpy()

    # kNN agreement for note type and diagnosis code in every layer
    note_type_agreement = []
    diagnosis_agreement = []
    for layer in range(13):
        agreement, used_labels = knn_agreement(layer_embeddings[layer], notes["note_type"], notes["patient_id"], "cosine")
        note_type_agreement.append(np.mean(agreement))
        agreement, used_labels = knn_agreement(layer_embeddings[layer], notes["diagnosis_code"], notes["patient_id"], "cosine")
        diagnosis_agreement.append(np.mean(agreement))

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(range(13), note_type_agreement, color=palette[0], label="Note type")
    ax.plot(range(13), diagnosis_agreement, color=palette[1], label="Diagnosis code")
    ax.axvline(main_layer, linestyle="--", color="grey") # Main layer
    ax.set_xlabel("FinBERT layer")
    ax.set_ylabel("kNN agreement")
    ax.set_xticks(range(13))
    ax.set_box_aspect(1)
    ax.legend()
    fig.savefig(results_folder + "/finbert_layers.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("FinBERT not run")

## 12. Clinical review <a class="anchor" id="section_12"></a>

Finally we wanted to know what actually separates the clusters. We split the main UMAP into clusters with k-means, and a clinician on the team read five random notes from each cluster and wrote down what set them apart: template, structure or content. On the real data this happened inside Acamedic, and only these categories were reported, never the text.

In [ ]:
# Clusters in the main UMAP
kmeans = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=10)
notes["cluster"] = kmeans.fit_predict(main_umap)

fig, ax = plt.subplots(figsize=(8, 8))
plot_by_label(ax, main_umap, notes["cluster"].astype(str), "cluster", "Clusters for the clinical review")
for cluster in range(number_of_clusters):
    centre = kmeans.cluster_centers_[cluster]
    ax.text(centre[0], centre[1], str(cluster), fontweight="bold") # Cluster number in the middle
fig.savefig(results_folder + "/clusters.png", dpi=150, bbox_inches="tight")
plt.show()

# Five random notes from each cluster
review_notes = notes.groupby("cluster").sample(n=notes_per_cluster, random_state=seed)

if data_source == "synthetic":
    for index in range(review_notes.shape[0]):
        print("Cluster %d, %s:"
              % (review_notes["cluster"].iloc[index], review_notes["note_type"].iloc[index]))
        print(review_notes["text"].iloc[index][:300])
        print()
else:
    # On the real data the notes stayed inside Acamedic, in a folder that is not in git
    os.makedirs("internal", exist_ok=True)
    review_notes[["cluster", "note_type", "text"]].to_csv("internal/review_notes.csv", index=False)

**What separated the clusters** (filled in by the clinician, no quotes from the notes)

| Cluster | What separates it (template, structure or content) |
|---|---|
| 0 | |
| 1 | |

## 13. Package versions <a class="anchor" id="section_13"></a>

In [ ]:
# Versions for the report
print("Python %s"
      % sys.version.split()[0])
for package in ["numpy", "pandas", "scikit-learn", "umap-learn", "matplotlib", "scipy", "requests", "torch", "transformers"]:
    print("%s %s"
          % (package, version(package)))